# Mammography lesion detection: training and evaluation

Research notebook for the YOLOv9 + DenseNet121 experiments. Run cells in order within each section; training sections represent distinct experiments and are optional when evaluating existing checkpoints. All metrics in the results table below come from the original run; cell outputs have been cleared for GitHub. This is research code, not a clinical tool.


## 1. Environment Setup

### Before running

Clone the repository, install `requirements.txt`, and open this notebook from the repository root or set `MAMMO_REPO_ROOT` to the checkout. Place processed CBIS-DDSM (`dataset.yaml`, `dataset_final.yaml`, `combined_test_metadata.csv`, `train/`, `val/`, `test/`) and INbreast (`dataset.yaml` and images/labels) outside Git; set `MAMMO_DATA_ROOT` to their parent directory. The expected subfolders are `YOLO_labelled_2c` and `inbreast_YOLO_labelled_2c`. Update the paths in the setup cell if your layout differs. The YAML files must reference your local image directories. The YOLOv9 fork under `model/` must include `train.py`, `val.py`, `models/`, `utils/`, and DenseNet backbone code. Train script must implement the 15-epoch freeze/unfreeze logic described in `model/README.md`; verify `--freeze` support for the initial experiment. Supply RadImageNet Keras weights separately; checkpoints and datasets are deliberately excluded from Git. CUDA is needed for practical training, while most evaluation code can also use CPU.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess

# Run from the checkout root, or set MAMMO_REPO_ROOT explicitly (useful in Colab).
notebook_cwd = Path.cwd().resolve()
repo_root = Path(os.environ.get("MAMMO_REPO_ROOT", notebook_cwd)).expanduser().resolve()
if not (repo_root / "model").is_dir() and (repo_root.parent / "model").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "model").is_dir():
    raise FileNotFoundError("Set MAMMO_REPO_ROOT to the cloned repository root containing model/.")
model_root = repo_root / "model"
data_root = Path(os.environ.get("MAMMO_DATA_ROOT", repo_root.parent)).expanduser().resolve()
cbis_root = data_root / "YOLO_labelled_2c"
inbreast_root = data_root / "inbreast_YOLO_labelled_2c"
cbis_yaml = cbis_root / "dataset.yaml"
final_yaml = cbis_root / "dataset_final.yaml"
inbreast_yaml = inbreast_root / "dataset.yaml"
train_script = model_root / "train.py"
val_script = model_root / "val.py"
rad_cfg = model_root / "yolov9-rad-densenet121.yaml"
img_cfg = model_root / "yolov9-imagenet-densenet121.yaml"
hyp_root = model_root / "hyps"
run_root = model_root / "runs" / "train"
rad_weights_dir = model_root / "radimagenet_weights"
if str(model_root) not in sys.path:
    sys.path.insert(0, str(model_root))

def run_yolo(script, *args):
    if not script.is_file():
        raise FileNotFoundError(f"Missing {script}; check model/ setup and README.")
    subprocess.run([sys.executable, str(script), *map(str, args)], cwd=repo_root, check=True)

print(f"Repo: {repo_root} | Data: {data_root}")


Import models and utilities from the modified YOLOv9 source in `model/`. The model and dataset files must be available before running training or evaluation.


In [ ]:
from models.yolo import Model
import torch
import cv2
import numpy as np
from utils.dataloaders import create_dataloader
from utils.general import check_dataset, check_img_size, colorstr, non_max_suppression
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, f1_score
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


In [ ]:
# Confirm local inputs before starting a long training run.
for path in (train_script, val_script, rad_cfg, img_cfg, cbis_yaml, final_yaml):
    if not path.exists():
        print(f"MISSING: {path}")


In [ ]:
# This notebook uses absolute paths and does not change the working directory.
print(f"YOLOv9 source: {model_root}")


## 2. Main Model Architecture

### 2.1 Developing a Model Better than Baseline

#### Model construction: RadImageNet DesnseNet121 YOLOv9

##### Direct name/shape mapping between Keras' DenseNet121 and torchvision.models for Keras to PyTorch conversion.

In [ ]:
# Step 1: Load the Keras RadImageNet DenseNet121 weights
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121 as KerasDenseNet121
import torchvision.models as tvm

KERAS_WEIGHTS_PATH = str(rad_weights_dir / "RadImageNet-DenseNet121_notop.h5")

keras_model = KerasDenseNet121(
    weights=KERAS_WEIGHTS_PATH,
    include_top=False,
    input_shape=(224, 224, 3),
)

# Step 2: Build a fresh torchvision DenseNet121 to receive the converted weights
torch_model = tvm.densenet121(weights=None)

# Step 3: Extract Keras conv/bn weights, in execution order
keras_sequence = []
for layer in keras_model.layers:
    cls_name = layer.__class__.__name__
    if cls_name == "Conv2D":
        w = layer.get_weights()
        assert len(w) == 1, f"Unexpected conv weight count for {layer.name}: {len(w)}"
        keras_sequence.append(("conv", w[0], layer.name))
    elif cls_name == "BatchNormalization":
        w = layer.get_weights()
        assert len(w) == 4, f"Unexpected BN weight count for {layer.name}: {len(w)}"
        keras_sequence.append(("bn", w, layer.name))

print(f"Keras: {len(keras_sequence)} weighted layers "
      f"({sum(1 for t,_,_ in keras_sequence if t=='conv')} conv, "
      f"{sum(1 for t,_,_ in keras_sequence if t=='bn')} bn)")

# Step 4: Extract torchvision conv/bn modules, in execution order
torch_sequence = []
for name, module in torch_model.named_modules():
    if isinstance(module, torch.nn.Conv2d):
        torch_sequence.append(("conv", module, name))
    elif isinstance(module, torch.nn.BatchNorm2d):
        torch_sequence.append(("bn", module, name))

print(f"Torch:  {len(torch_sequence)} weighted layers "
      f"({sum(1 for t,_,_ in torch_sequence if t=='conv')} conv, "
      f"{sum(1 for t,_,_ in torch_sequence if t=='bn')} bn)")

assert len(keras_sequence) == len(torch_sequence), (
    f"Layer count mismatch: Keras={len(keras_sequence)}, Torch={len(torch_sequence)} — "
    "stop here and inspect both sequences before proceeding, architectures may differ."
)

# Step 5: Copy weights position-by-position, verifying type/shape at each step
for (k_type, k_weight, k_name), (t_type, t_module, t_name) in zip(keras_sequence, torch_sequence):
    assert k_type == t_type, f"Type mismatch: {k_name} (keras, {k_type}) vs {t_name} (torch, {t_type})"

    if k_type == "conv":
        # Keras kernel: (H, W, in_ch, out_ch) -> PyTorch: (out_ch, in_ch, H, W)
        torch_weight = np.transpose(k_weight, (3, 2, 0, 1))
        assert torch_weight.shape == tuple(t_module.weight.shape), (
            f"Shape mismatch at {k_name}/{t_name}: {torch_weight.shape} vs {tuple(t_module.weight.shape)}"
        )
        with torch.no_grad():
            t_module.weight.copy_(torch.from_numpy(torch_weight))

    elif k_type == "bn":
        gamma, beta, moving_mean, moving_variance = k_weight
        with torch.no_grad():
            t_module.weight.copy_(torch.from_numpy(gamma))
            t_module.bias.copy_(torch.from_numpy(beta))
            t_module.running_mean.copy_(torch.from_numpy(moving_mean))
            t_module.running_var.copy_(torch.from_numpy(moving_variance))

print("Conversion complete — all layers copied by matched sequence position.")

# Step 6: Immediate sanity check
print("\n--- BatchNorm running_var sanity check ---")
found_issue = False
for name, module in torch_model.named_modules():
    if isinstance(module, torch.nn.BatchNorm2d):
        rv = module.running_var
        if rv.min().item() < 1e-6 or rv.max().item() > 1e4:
            print(f"SUSPICIOUS: {name} min={rv.min().item():.4f} max={rv.max().item():.4f}")
            found_issue = True
if not found_issue:
    print("All running_var values within a sane range.")

# Step 7: Save model, ready to plug into existing RadDenseNet121Backbone loading code
rad_weights_dir.mkdir(parents=True, exist_ok=True)
torch.save(torch_model.state_dict(), rad_weights_dir / "radimagenet_densenet121_selfconverted.pt")
print("Saved:", rad_weights_dir / "radimagenet_densenet121_selfconverted.pt")

##### Verify PyTorch model construction

In [ ]:
# Load model
cfg = str(rad_cfg)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Model(cfg, ch=3, nc=2).to(device).eval()
backbone = model.model[0]  # the RadDenseNet121Backbone layer, index 0 per your YAML

# Load a sample mammogram
img_path = str(cbis_root / "train/images/Calc-Training_P_00005_RIGHT_CC.png")
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB) # Convert BGR → RGB
img = img_rgb.astype(np.float32) / 255.0 # Convert to float and normalise
img = np.transpose(img, (2, 0, 1)) # HWC → CHW

# Add batch dimension
x = torch.from_numpy(img).unsqueeze(0).to(device)

print("Input shape:", x.shape)

# Forward inference
with torch.no_grad():
    y = model(x)
    feats = backbone(x)
for i, f in enumerate(feats):
    print(f"P{i}: mean={f.mean().item():.4f}, std={f.std().item():.4f}, max={f.max().item():.2f}")
pred = y[0]
print("Prediction output:", pred.shape)
print("Forward inference successful!")

#### Model training

In [ ]:
# Check preprocessing outputs and optional pretrained weights.
for path in (cbis_root, rad_weights_dir):
    print(path, "present" if path.exists() else "missing (supply locally)")


In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "15",
    "--data", cbis_yaml, "--cfg", rad_cfg,
    "--hyp", hyp_root / "hyp.overfit-test.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "statistical_power_test_rad_densenet121", "--freeze", "1")


### 2.2 Scaling Up: Developing a Model that Overfits

#### Train RadImageNet DenseNet121 YOLOv9. Freeze backbone on first 15 epochs. Fine-tuning after.

In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "100",
    "--data", cbis_yaml, "--cfg", rad_cfg,
    "--hyp", hyp_root / "hyp.overfit-test.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "overfit_test_rad_densenet121")


### 2.3 Regularising and Tuning Hyperparameters

#### Tuning Variant A: baseline

In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "100",
    "--data", cbis_yaml, "--cfg", rad_cfg,
    "--hyp", hyp_root / "hyp.transfer-low-lr.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "tune_A_baseline")


#### Tuning Variant B: removed COCO-style augmentation

In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "100",
    "--data", cbis_yaml, "--cfg", rad_cfg,
    "--hyp", hyp_root / "hyp.tune-B-noaug.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "tune_B_noaug")


In [ ]:
# Historical resume run; only execute if this checkpoint is available.
resume_ckpt = run_root / "tune_B_noaug2" / "weights/last.pt"
run_yolo(train_script, "--resume", resume_ckpt)


#### Tuning Variant C: reduce relative weight on classification to free up gradient signal for localisation

In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "100",
    "--data", cbis_yaml, "--cfg", rad_cfg,
    "--hyp", hyp_root / "hyp.tune-C-clsweight.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "tune_C_clsweight")


In [ ]:
# Historical resume run; only execute if this checkpoint is available.
resume_ckpt = run_root / "tune_C_clsweight" / "weights/last.pt"
run_yolo(train_script, "--resume", resume_ckpt)


#### Tuning Variant D: combined effect of Variants B and C

In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "100",
    "--data", cbis_yaml, "--cfg", rad_cfg,
    "--hyp", hyp_root / "hyp.tune-D-combined.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "tune_D_combined")


### 2.4 Best model trained on full dataset

#### Train RadImageNet DenseNet121 YOLOv9. Freeze backbone on first 15 epochs. Fine-tuning after.

In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "68",
    "--data", final_yaml, "--cfg", rad_cfg,
    "--hyp", hyp_root / "hyp.tune-C-clsweight.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "final_radimagenet_densenet121")


## 3. ImageNet-DenseNet121 model

### Model Construction: ImageNet DesnseNet121 YOLOv9 model

In [ ]:
# Load model
cfg = str(img_cfg)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Model(cfg, ch=3, nc=2).to(device).eval()
backbone = model.model[0]  # the ImageNetDenseNet121Backbone layer, index 0 per your YAML

# Load a sample mammogram
img_path = str(cbis_root / "train/images/Calc-Training_P_00005_RIGHT_CC.png")
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB) # Convert BGR → RGB
img = img_rgb.astype(np.float32) / 255.0 # Convert to float and normalise
img = np.transpose(img, (2, 0, 1)) # HWC → CHW

# Add batch dimension
x = torch.from_numpy(img).unsqueeze(0).to(device)

print("Input shape:", x.shape)

# Forward inference
with torch.no_grad():
    y = model(x)
pred = y[0]
print("Prediction output:", pred.shape)
print("Forward inference successful!")

### Train ImageNet DenseNet121 YOLOv9. Freeze backbone on first 15 epochs. Fine-tuning after.

In [ ]:
run_yolo(train_script,
    "--img", "1024", "--batch", "4", "--epochs", "68",
    "--data", final_yaml, "--cfg", img_cfg,
    "--hyp", hyp_root / "hyp.tune-C-clsweight.yaml", "--weights", "",
    "--optimizer", "AdamW", "--name", "imagenet_densenet121")


## 4. Results

### Load validation dataset

In [ ]:
data_dict = check_dataset(str(cbis_yaml))

val_loader = create_dataloader(
    data_dict['val'],
    imgsz=1024,
    batch_size=4,
    stride=32,
    single_cls=False,
    hyp=None,
    cache=None,
    rect=True,
    rank=-1,
    workers=8,
    pad=0.5,
    prefix=colorstr('val: ')
)[0]

### Function to calculate AUC-ROC, Accuracy and F1-score

In [ ]:
def get_image_level_scores_with_paths(
    model,
    dataloader,
    device,
    malignant_class_idx=1,
    conf_thres=0.001,
    iou_thres=0.5
):
    image_scores = []
    image_labels = []
    image_paths = []

    model.eval()

    with torch.no_grad():
        for imgs, targets, paths, shapes in dataloader:

            imgs = imgs.to(device).float() / 255

            raw_preds = model(imgs)
            preds = raw_preds[0] if isinstance(raw_preds, (tuple, list)) else raw_preds

            preds = non_max_suppression(
                preds,
                conf_thres=conf_thres,
                iou_thres=iou_thres,
                multi_label=True
            )

            batch_size = imgs.shape[0]

            for i in range(batch_size):

                img_targets = targets[targets[:, 0] == i]

                # Image is malignant if ANY GT lesion is malignant
                gt_label = int(
                    (img_targets[:, 1] == malignant_class_idx).any().item()
                ) if len(img_targets) else 0

                det = preds[i]

                if det is not None and len(det):

                    malignant_dets = det[
                        det[:, 5] == malignant_class_idx
                    ]

                    score = (
                        malignant_dets[:, 4].max().item()
                        if len(malignant_dets)
                        else 0.0
                    )

                else:
                    score = 0.0

                image_labels.append(gt_label)
                image_scores.append(score)
                image_paths.append(paths[i])

    return (
        np.array(image_scores),
        np.array(image_labels),
        image_paths
    )

### Baseline model performance

In [ ]:
# Load model
model_name = "statistical_power_test_rad_densenet121"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(run_root / model_name / "weights/best.pt", map_location="cpu", weights_only=False)
model = ckpt['ema'] or ckpt['model']
model = model.float().to(device).eval()

In [ ]:
# Calculate AUC-ROC, Accuracy and F1-score
scores, labels, _ = get_image_level_scores_with_paths(model, val_loader, device, malignant_class_idx=1)

print(f"Model: {model_name}")
print(f"Positive (malignant) images: {labels.sum()} / {len(labels)}")
auc = roc_auc_score(labels, scores)
print(f"Image-level AUC-ROC: {auc:.4f}")

fpr, tpr, thresholds = roc_curve(labels, scores)

# Select threshold on VAL set using Youden's J statistic (maximizes sensitivity + specificity - 1)
youden_j = tpr - fpr
best_threshold = thresholds[np.argmax(youden_j)]
print(f"Selected threshold (Youden's J): {best_threshold:.4f}")

# Apply that fixed threshold to VAL set
val_preds_binary = (scores >= best_threshold).astype(int)
acc = accuracy_score(labels, val_preds_binary)
f1 = f1_score(labels, val_preds_binary)
print(f"Val Accuracy: {acc:.4f}, Val F1: {f1:.4f}")

### Evaluating Model that Overfits using Loss Curves

In [ ]:
Overfit_results_df = pd.read_csv(run_root / "overfit_test_rad_densenet121/results.csv")
Overfit_results_df.columns = Overfit_results_df.columns.str.strip()
print(Overfit_results_df.columns)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Box loss
axes[0].plot(Overfit_results_df["train/box_loss"], label="Train")
axes[0].plot(Overfit_results_df["val/box_loss"], label="Validation")
axes[0].set_title("Box Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

# Classification loss
axes[1].plot(Overfit_results_df["train/cls_loss"], label="Train")
axes[1].plot(Overfit_results_df["val/cls_loss"], label="Validation")
axes[1].set_title("Classification Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

# DFL loss
axes[2].plot(Overfit_results_df["train/dfl_loss"], label="Train")
axes[2].plot(Overfit_results_df["val/dfl_loss"], label="Validation")
axes[2].set_title("DFL Loss")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.show()

### Results of Hyperparameter Variants

In [ ]:
results_summary = []

for name in ["tune_A_baseline", "tune_B_noaug", "tune_C_clsweight", "tune_D_combined"]:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    ckpt = torch.load(run_root / name / "weights/best.pt", map_location="cpu", weights_only=False)
    model = ckpt['ema'] or ckpt['model']
    model = model.float().to(device).eval()

    scores, labels, _ = get_image_level_scores_with_paths(model, val_loader, device, malignant_class_idx=1)
    auc = roc_auc_score(labels, scores)
    fpr, tpr, thresholds = roc_curve(labels, scores)

    youden_j = tpr - fpr
    best_threshold = thresholds[np.argmax(youden_j)]

    test_preds_binary = (scores >= best_threshold).astype(int)
    acc = accuracy_score(labels, test_preds_binary)
    f1 = f1_score(labels, test_preds_binary)

    results_summary.append({"variant": name, "val_auc": auc, "val_Youdens_J": best_threshold, "val_accuracy": acc, "val_f1": f1})

print(pd.DataFrame(results_summary).sort_values("val_auc", ascending=False))

#### Best epoch count for training

In [ ]:
df = pd.read_csv(run_root / "tune_C_clsweight/results.csv")
df.columns = df.columns.str.strip()

df['val_loss_total'] = df['val/box_loss'] + df['val/cls_loss'] + df['val/dfl_loss']
best_epoch_by_loss = df.loc[df['val_loss_total'].idxmin(), 'epoch']
print(f"Best epoch by combined val loss: {best_epoch_by_loss}")

### Final RadImageNet-DenseNet121 Model Performance on Unseen Test Set

In [ ]:
# Load test dataset
final_data_dict = check_dataset(str(final_yaml))

final_test_loader = create_dataloader(
    final_data_dict['test'],
    imgsz=1024,
    batch_size=4,
    stride=32,
    single_cls=False,
    hyp=None,
    cache=None,
    rect=True,
    rank=-1,
    workers=8,
    pad=0.5,
    prefix=colorstr('test: ')
)[0]

print("Test dataset size:", len(final_test_loader.dataset))

In [ ]:
# Load final model
final_model_name = "final_radimagenet_densenet121"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(run_root / final_model_name / "weights/last.pt", map_location="cpu", weights_only=False)
final_model = ckpt['ema'] or ckpt['model']
final_model = final_model.float().to(device).eval()

In [ ]:
# Threshold on VAL set of best model (Variant C)
tuning_best_threshold = 0.046104

# Run inference on held-out TEST set
final_test_scores, final_test_labels, _ = get_image_level_scores_with_paths(
    final_model,
    final_test_loader,
    device,
    malignant_class_idx=1
)

print("=" * 60)
print("FINAL HELD-OUT TEST RESULTS")
print("=" * 60)

print(f"Model: {final_model_name}")
print(f"Positive (malignant) images: {final_test_labels.sum()} / {len(final_test_labels)}")
final_test_auc = roc_auc_score(final_test_labels, final_test_scores)
print(f"Image-level AUC-ROC: {final_test_auc:.4f}")

# Apply fixed validation-selected threshold
final_test_preds_binary = (
    final_test_scores >= tuning_best_threshold
).astype(int)
final_test_acc = accuracy_score(final_test_labels, final_test_preds_binary)
final_test_f1 = f1_score(final_test_labels, final_test_preds_binary)
print(f"Fixed threshold from Variant C VAL set: {tuning_best_threshold:.6f}")
print(f"Test Accuracy: {final_test_acc:.4f}, Test F1: {final_test_f1:.4f}")

### ImageNet-DenseNet121 Model Performance

In [ ]:
# Load ImageNet model
model_name = "imagenet_densenet121"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(run_root / model_name / "weights/last.pt", map_location="cpu", weights_only=False)
model = ckpt['ema'] or ckpt['model']
model = model.float().to(device).eval()

In [ ]:
# Threshold on VAL set of best model (Variant C)
tuning_best_threshold = 0.046104

# Run inference on held-out TEST set
test_scores, test_labels, _ = get_image_level_scores_with_paths(
    model,
    final_test_loader,
    device,
    malignant_class_idx=1
)

print("=" * 60)
print("FINAL HELD-OUT TEST RESULTS")
print("=" * 60)

print(f"Model: {model_name}")
print(f"Positive (malignant) images: {test_labels.sum()} / {len(test_labels)}")
test_auc = roc_auc_score(test_labels, test_scores)
print(f"Image-level AUC-ROC: {test_auc:.4f}")

# Apply fixed validation-selected threshold
test_preds_binary = (
    test_scores >= tuning_best_threshold
).astype(int)
test_acc = accuracy_score(test_labels, test_preds_binary)
test_f1 = f1_score(test_labels, test_preds_binary)
print(f"Fixed threshold from Variant C VAL set: {tuning_best_threshold:.6f}")
print(f"Test Accuracy: {test_acc:.4f}, Test F1: {test_f1:.4f}")

### Held-out CBIS-DDSM results (recorded run)

| Model | AUC-ROC (image) | Accuracy | F1 | mAP@0.5 | mAP@0.5:0.95 |
| --- | ---: | ---: | ---: | ---: | ---: |
| RadImageNet DenseNet121 | 0.6288 | 0.5779 | 0.5512 | 0.199 | 0.0913 |
| ImageNet DenseNet121 | 0.6463 | 0.5622 | 0.5805 | 0.208 | 0.0891 |

DeLong p = **0.3027** for paired image-level AUCs. These scores are descriptive; the comparison did not show a statistically significant difference. The threshold 0.046104 was selected on validation data for the RadImageNet tuning variant and held fixed on the test set. See the code below for evaluation details.


#### DeLong's test

##### Test setup

In [ ]:
from scipy import stats

def compute_midrank(x):
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    T2 = np.empty(N, dtype=float)
    T2[J] = T
    return T2

def fastDeLong(predictions_sorted_transposed, label_1_count):
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]
    k = predictions_sorted_transposed.shape[0]

    tx = np.empty([k, m], dtype=float)
    ty = np.empty([k, n], dtype=float)
    tz = np.empty([k, m + n], dtype=float)
    for r in range(k):
        tx[r, :] = compute_midrank(positive_examples[r, :])
        ty[r, :] = compute_midrank(negative_examples[r, :])
        tz[r, :] = compute_midrank(predictions_sorted_transposed[r, :])
    aucs = tz[:, :m].sum(axis=1) / m / n - float(m + 1) / 2 / n
    v01 = (tz[:, :m] - tx[:, :]) / n
    v10 = 1 - (tz[:, m:] - ty[:, :]) / m
    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n
    return aucs, delongcov

def calc_pvalue(aucs, sigma):
    l = np.array([[1, -1]])
    z = np.abs(np.diff(aucs)) / np.sqrt(np.dot(np.dot(l, sigma), l.T))
    return float(2 * (1 - stats.norm.cdf(z))[0][0])

def delong_roc_test(ground_truth, predictions_one, predictions_two):
    """
    ground_truth: (N,) binary array, same order for both models
    predictions_one, predictions_two: (N,) continuous scores from the two models,
        evaluated on the SAME test images, in the SAME order
    Returns: (auc_model1, auc_model2, p_value)
    """
    order = np.argsort(-ground_truth)
    label_1_count = int(ground_truth.sum())
    predictions_sorted_transposed = np.vstack((predictions_one, predictions_two))[:, order]
    aucs, delongcov = fastDeLong(predictions_sorted_transposed, label_1_count)
    p = calc_pvalue(aucs, delongcov)
    return aucs[0], aucs[1], p

##### Run test

In [ ]:
scores_rad, labels_rad, paths_rad = get_image_level_scores_with_paths(final_model, final_test_loader, device)
scores_img, labels_img, paths_img = get_image_level_scores_with_paths(model, final_test_loader, device)
assert paths_rad == paths_img, "Test image order mismatch; pair predictions by image path."
assert np.array_equal(labels_rad, labels_img), "Ground truth mismatch across model predictions."
auc_rad, auc_img, p_value = delong_roc_test(labels_rad, scores_rad, scores_img)
print(f"RadImageNet AUC: {auc_rad:.4f}")
print(f"ImageNet AUC:    {auc_img:.4f}")
print(f"DeLong p-value:  {p_value:.4f}")


#### Test set mAP evaluation

##### RadImageNet-DenseNet121

In [ ]:
run_yolo(val_script, "--data", final_yaml,
    "--weights", run_root / "final_radimagenet_densenet121" / "weights/last.pt",
    "--img", "1024", "--batch", "4", "--task", "test",
    "--name", "test_eval_radimagenet_densenet121", "--verbose")


##### ImageNet-DenseNet121

In [ ]:
run_yolo(val_script, "--data", final_yaml,
    "--weights", run_root / "imagenet_densenet121" / "weights/last.pt",
    "--img", "1024", "--batch", "4", "--task", "test",
    "--name", "test_eval_imagenet_densenet121", "--verbose")


#### RadImageNet vs ImageNet table

In [ ]:
# Recorded values from the original run; rerun preceding cells to reproduce.
data = {
    'Model': ['final_radimagenet_densenet121', 'imagenet_densenet121'],
    'Image-level AUC-ROC': [0.6288, 0.6463],
    'Test Accuracy': [0.5779, 0.5622],
    'Test F1-score': [0.5512, 0.5805],
    'mAP@0.5': [0.199, 0.208],
    'mAP@0.5:0.95': [0.0913, 0.0891]
}

df = pd.DataFrame(data)
df

### External validation on INbreast (recorded run)

The final RadImageNet model reached image-level AUC-ROC **0.7549**, accuracy **0.6939**, F1 **0.5914** at the CBIS-DDSM validation-selected threshold, and lesion mAP@0.5 **0.00792** (mAP@0.5:0.95 **0.00341**). The very low detection mAP limits claims about cross-dataset lesion localization. Check the INbreast label conversion and annotation granularity before interpreting its lesion counts as directly comparable to CBIS-DDSM.


##### AUC, Accuracy, F1-score

In [ ]:
# Load final model
final_model_name = "final_radimagenet_densenet121"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load(run_root / final_model_name / "weights/last.pt", map_location="cpu", weights_only=False)
final_model = ckpt['ema'] or ckpt['model']
final_radimagenet_model = final_model.float().to(device).eval()

In [ ]:
# Load INbreast dataset
inbreast_data_dict = check_dataset(str(inbreast_yaml))

# Use the stride of the loaded final model
stride = int(final_radimagenet_model.stride.max())

# Ensure image size is compatible with model stride
imgsz = check_img_size(1024, s=stride)

inbreast_loader = create_dataloader(
    inbreast_data_dict['test'],
    imgsz=imgsz,
    batch_size=4,
    stride=stride,
    single_cls=False,
    hyp=None,
    augment=False,
    cache=None,
    rect=True,
    rank=-1,
    workers=8,
    pad=0.5,
    shuffle=False,
    prefix=colorstr("INbreast test: ")
)[0]

print(inbreast_data_dict)
print("Model stride:", stride)
print("Image size:", imgsz)
print("Number of images:", len(inbreast_loader.dataset))

In [ ]:
# Threshold on VAL set of best model (Variant C)
tuning_best_threshold = 0.046104

# Run inference on INbreast dataset
inbreast_scores, inbreast_labels, _ = get_image_level_scores_with_paths(
    final_model,
    inbreast_loader,
    device,
    malignant_class_idx=1
)

print("=" * 60)
print("RESULTS ON INBREAST DATASET")
print("=" * 60)

print(f"Model: {final_model_name}")
print(f"Positive (malignant) images: {inbreast_labels.sum()} / {len(inbreast_labels)}")
inbreast_auc = roc_auc_score(inbreast_labels, inbreast_scores)
print(f"Image-level AUC-ROC: {inbreast_auc:.4f}")

# Apply fixed validation-selected threshold
inbreast_preds_binary = (
    inbreast_scores >= tuning_best_threshold
).astype(int)
inbreast_acc = accuracy_score(inbreast_labels, inbreast_preds_binary)
inbreast_f1 = f1_score(inbreast_labels, inbreast_preds_binary)
print(f"Fixed threshold from Variant C VAL set: {tuning_best_threshold:.6f}")
print(f"Test Accuracy: {inbreast_acc:.4f}, Test F1: {inbreast_f1:.4f}")

##### mAP evaluation

In [ ]:
run_yolo(val_script, "--data", inbreast_yaml,
    "--weights", run_root / "final_radimagenet_densenet121" / "weights/last.pt",
    "--img", "1024", "--batch", "4", "--task", "test",
    "--name", "inbreast_eval_radimagenet_densenet121", "--verbose")


### Error Analysis

In [ ]:
from tqdm import tqdm
from utils.general import box_iou

#### Ground Truth of the detections

In [ ]:
def xywhn_to_xyxy(boxes, w, h):
    """
    YOLO normalized:
    x_center, y_center, width, height
    ->
    pixel xyxy
    """
    boxes = boxes.clone()

    x = boxes[:, 0] * w
    y = boxes[:, 1] * h
    bw = boxes[:, 2] * w
    bh = boxes[:, 3] * h

    out = torch.zeros_like(boxes)

    out[:, 0] = x - bw / 2
    out[:, 1] = y - bh / 2
    out[:, 2] = x + bw / 2
    out[:, 3] = y + bh / 2

    return out

In [ ]:
CONF_THRES = 0.046104   # validation-selected threshold
IOU_THRES = 0.5

results = []

final_model.eval()

with torch.no_grad():

    for imgs, targets, paths, shapes in tqdm(final_test_loader):

        # Move images to same device as model
        imgs = imgs.to(device, non_blocking=True)
        imgs = imgs.float() / 255.0

        # Move targets too if they're used in GPU computations
        targets = targets.to(device)

        # Forward pass
        pred = final_model(imgs)

        # Depending on YOLOv9 output format
        if isinstance(pred, (list, tuple)):
            pred = pred[0]

        preds = non_max_suppression(
            pred,
            conf_thres=CONF_THRES,
            iou_thres=0.5
        )

        for i, detections in enumerate(preds):

            img_path = paths[i]

            _, _, H, W = imgs.shape

            # Ground truth belonging to this image
            gt = targets[targets[:, 0] == i].clone()

            if len(gt) == 0:
                continue

            gt_classes = gt[:, 1].long()
            gt_boxes = xywhn_to_xyxy(gt[:, 2:6], W, H)

            # No predictions = every GT lesion is FN
            if detections is None or len(detections) == 0:

                for j in range(len(gt_boxes)):

                    box = gt[:, 2:6][j]

                    results.append({
                        "image": img_path,
                        "gt_class": int(gt_classes[j]),
                        "detected": False,
                        "max_iou": 0.0,
                        "gt_width": float(box[2]),
                        "gt_height": float(box[3]),
                        "gt_area": float(box[2] * box[3])
                    })

                continue

            pred_boxes = detections[:, :4]
            pred_classes = detections[:, 5].long()

            ious = box_iou(gt_boxes, pred_boxes)

            for j in range(len(gt_boxes)):

                # Only compare against predictions of same class
                same_class = pred_classes == gt_classes[j]

                if same_class.any():

                    matching_ious = ious[j][same_class]
                    max_iou = matching_ious.max().item()

                else:
                    max_iou = 0.0

                detected = max_iou >= IOU_THRES

                box = gt[:, 2:6][j]

                results.append({
                    "image": img_path,
                    "gt_class": int(gt_classes[j]),
                    "detected": detected,
                    "max_iou": max_iou,
                    "gt_width": float(box[2]),
                    "gt_height": float(box[3]),
                    "gt_area": float(box[2] * box[3])
                })

error_df = pd.DataFrame(results)
error_df.head()

In [ ]:
# Save analysis alongside local outputs; do not commit patient-level outputs.
analysis_output = data_root / "error_analysis.csv"
error_df.to_csv(analysis_output, index=False)


#### Get lesion-size analysis

In [ ]:
error_df["size_group"] = pd.qcut(
    error_df["gt_area"],
    q=4,
    labels=["Smallest Q1", "Q2", "Q3", "Largest Q4"]
)

In [ ]:
size_analysis = (
    error_df
    .groupby("size_group", observed=False)
    .agg(
        lesions=("detected", "count"),
        detected=("detected", "sum")
    )
)

size_analysis["false_negatives"] = (
    size_analysis["lesions"] -
    size_analysis["detected"]
)

size_analysis["FN_rate"] = (
    size_analysis["false_negatives"] /
    size_analysis["lesions"]
)

print(size_analysis)

#### Generate TP, TN, FP and FN for every mammogram in test set

In [ ]:
scores, labels, paths = get_image_level_scores_with_paths(
    final_model,
    final_test_loader,
    device,
    malignant_class_idx=1
)

BEST_THRESHOLD = 0.046104
predicted_labels = (scores >= BEST_THRESHOLD).astype(int)

In [ ]:
from pathlib import Path

error_df = pd.DataFrame({
    "model_image_path": paths,
    "true_label": labels,
    "score": scores,
    "predicted_label": predicted_labels
})

def classify_result(row):

    if row["true_label"] == 1 and row["predicted_label"] == 1:
        return "TP"

    elif row["true_label"] == 0 and row["predicted_label"] == 0:
        return "TN"

    elif row["true_label"] == 0 and row["predicted_label"] == 1:
        return "FP"

    elif row["true_label"] == 1 and row["predicted_label"] == 0:
        return "FN"


error_df["result"] = error_df.apply(
    classify_result,
    axis=1
)

print(error_df.head())

print("\nPrediction outcomes:")
print(error_df["result"].value_counts())

#### Prepare CBIS-DDSM metadata to analyse results

In [ ]:
error_df["image_key"] = error_df["model_image_path"].apply(
    lambda x: Path(str(x)).stem
)

print(error_df[[
    "model_image_path",
    "image_key"
]].head())

In [ ]:
combined_test_metadata_df = pd.read_csv(cbis_root / "combined_test_metadata.csv")

In [ ]:
import re

def extract_image_key(path):
    path = str(path)
    match = re.search(
        r'(Mass|Calc)-Test_P_\d+_(LEFT|RIGHT)_(CC|MLO)',
        path
    )
    if match:
        return match.group(0)
    return None

combined_test_metadata_df["image_key"] = combined_test_metadata_df["full_img_path"].apply(
    extract_image_key
)

print(combined_test_metadata_df[[
    "full_img_path",
    "image_key",
    "breast_density",
    "subtlety",
    "abnormality type"
]].head())

In [ ]:
error_keys = set(error_df["image_key"].dropna())
metadata_keys = set(combined_test_metadata_df["image_key"].dropna())

matched_keys = error_keys & metadata_keys

print("YOLO test images:", len(error_keys))
print("Metadata images:", len(metadata_keys))
print("Matched images:", len(matched_keys))
print("Unmatched YOLO images:", len(error_keys - metadata_keys))

In [ ]:
image_metadata = (
    combined_test_metadata_df
    .dropna(subset=["image_key"])
    .groupby("image_key", as_index=False)
    .agg({
        "breast_density": "first",
        "subtlety": "min",
        "abnormality type": lambda x: ", ".join(
            sorted(set(x.dropna().astype(str)))
        )
    })
)

analysis_df = error_df.merge(
    image_metadata,
    on="image_key",
    how="left"
)

density_df = analysis_df.dropna(subset=["breast_density"]).copy()

print("Images with density data:", len(density_df))
print("Images excluded due to missing density:",
      analysis_df["breast_density"].isna().sum())

#### Get breast density analysis

In [ ]:
malignant_by_density = (
    density_df[density_df["true_label"] == 1]
    .groupby("breast_density")
    .agg(
        malignant_images=("true_label", "size"),
        false_negatives=("result", lambda x: (x == "FN").sum())
    )
)

malignant_by_density["FN_rate"] = (
    malignant_by_density["false_negatives"]
    / malignant_by_density["malignant_images"]
)

display(malignant_by_density)

In [ ]:
benign_by_density = (
    density_df[density_df["true_label"] == 0]
    .groupby("breast_density")
    .agg(
        benign_images=("true_label", "size"),
        false_positives=("result", lambda x: (x == "FP").sum())
    )
)

benign_by_density["FP_rate"] = (
    benign_by_density["false_positives"]
    / benign_by_density["benign_images"]
)

display(benign_by_density)

#### Get subtlety analysis

In [ ]:
fn_by_subtlety = (
    analysis_df[
        analysis_df["true_label"] == 1
    ]
    .dropna(subset=["subtlety"])
    .groupby("subtlety")
    .agg(
        malignant_images=("true_label", "size"),
        false_negatives=("result", lambda x: (x == "FN").sum())
    )
)

fn_by_subtlety["FN_rate"] = (
    fn_by_subtlety["false_negatives"]
    / fn_by_subtlety["malignant_images"]
)

display(fn_by_subtlety)

#### Get abnormality analysis

In [ ]:
abnormality_errors = (
    analysis_df
    .groupby("abnormality type")
    .agg(
        images=("result", "size"),
        false_positives=("result", lambda x: (x == "FP").sum()),
        false_negatives=("result", lambda x: (x == "FN").sum())
    )
)

display(abnormality_errors)

## 5. Code attribution and references


Ansel, J. et al. (2024) 'PyTorch 2: Faster Machine Learning Through Dynamic Python Bytecode Transformation and Graph Compilation', *in 29th ACM International Conference on Architectural Support for Programming Languages and Operating Systems, Volume 2 (ASPLOS '24)*. ACM. Available at: https://doi.org/10.1145/3620665.3640366.


Lee, R.S. *et al.* (2017) *Curated Breast Imaging Subset of Digital Database for Screening Mammography (CBIS-DDSM) [Data set], The Cancer Imaging Archive*. Available at: https://www.cancerimagingarchive.net/collection/cbis-ddsm/ (Accessed: 03 June 2026).


Mei, X. *et al.* (2022) 'RadImageNet: An Open Radiologic Deep Learning Research Dataset for Effective Transfer Learning', *Radiology: Artificial Intelligence, 0(ja)*. Available at: doi:10.1148/ryai.210315.